In [ ]:
!pip install groq gradio python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.3 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from groq import Groq

# Get key from Colab secrets
api_key = userdata.get('GROQ_API_KEY')

# Connect to Groq
client = Groq(api_key=api_key)

# Test
chat = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Say hello in one sentence!"}
    ]
)

print(chat.choices[0].message.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [ ]:
from google.colab import userdata
from groq import Groq
import json
import datetime

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

# ===== TOOLS =====
def get_current_date():
    return datetime.datetime.now().strftime("%A, %B %d, %Y")

def calculate(expression):
    try:
        result = eval(expression)
        return f"Result: {result}"
    except:
        return "Invalid calculation"

def generate_email(client_name, purpose, tone):
    return f"""
Subject: Follow Up — {purpose}

Dear {client_name},

I hope this message finds you well.
I wanted to follow up regarding {purpose}.
Please let me know if you need anything from our side.

Best regards,
[Your Name]
"""

def execute_tool(tool_name, tool_args):
    if tool_name == "get_current_date":
        return get_current_date()
    elif tool_name == "calculate":
        return calculate(tool_args["expression"])
    elif tool_name == "generate_email":
        return generate_email(
            tool_args["client_name"],
            tool_args["purpose"],
            tool_args["tone"]
        )

# ===== TOOL DESCRIPTIONS =====
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_date",
            "description": "Get todays current date",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Calculate a math expression like profit, revenue",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression like 500*12"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_email",
            "description": "Generate a professional business email",
            "parameters": {
                "type": "object",
                "properties": {
                    "client_name": {"type": "string"},
                    "purpose":     {"type": "string"},
                    "tone":        {"type": "string"}
                },
                "required": ["client_name", "purpose", "tone"]
            }
        }
    }
]

# ===== SYSTEM PROMPT =====
system_prompt = """
You are CopilotAI, a smart business assistant.
You have access to tools — always use them when needed.
After getting tool results, always give a clean final answer to the user.
"""

# ===== AGENT LOOP =====
def chat_with_agent(user_message):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message}
    ]

    while True:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        finish_reason    = response.choices[0].finish_reason

        # Agent wants to use a tool
        if finish_reason == "tool_calls" and response_message.tool_calls:

            messages.append({
                "role": "assistant",
                "content": response_message.content or "",
                "tool_calls": [
                    {
                        "id":       tc.id,
                        "type":     "function",
                        "function": {
                            "name":      tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    }
                    for tc in response_message.tool_calls
                ]
            })

            for tc in response_message.tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)

                print(f"🔧 Using tool: {tool_name} with {tool_args}")

                result = execute_tool(tool_name, tool_args)

                messages.append({
                    "role":         "tool",
                    "tool_call_id": tc.id,
                    "name":         tool_name,
                    "content":      str(result)
                })

        # Agent gives final answer
        else:
            return response_message.content

# ===== TEST =====
print(chat_with_agent("What is today's date?"))
print("---")
print(chat_with_agent("If I sell 150 products at $45 each what is my total revenue?"))
print("---")
print(chat_with_agent("Write a follow up email to John about his pending invoice, keep it formal"))

🔧 Using tool: get_current_date with None
Today's date is Sunday, May 10, 2026.
---


BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=calculate={"expression": "150*45"}</function>'}}

In [ ]:
from google.colab import userdata
from groq import Groq
import json
import datetime

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

# ===== TOOLS =====
def get_current_date():
    return datetime.datetime.now().strftime("%A, %B %d, %Y")

def calculate(expression):
    try:
        result = eval(expression)
        return f"{result}"
    except:
        return "Invalid calculation"

def generate_email(client_name, purpose, tone):
    return f"""Subject: Follow Up — {purpose}

Dear {client_name},

I hope this message finds you well.
I wanted to follow up regarding {purpose}.
Please let me know if you need anything from our side.

Best regards,
[Your Name]"""

def execute_tool(tool_name, tool_args):
    if tool_name == "get_current_date":
        return get_current_date()
    elif tool_name == "calculate":
        return calculate(tool_args["expression"])
    elif tool_name == "generate_email":
        return generate_email(
            tool_args["client_name"],
            tool_args["purpose"],
            tool_args["tone"]
        )
    return "Tool not found"

# ===== SYSTEM PROMPT — Agent decides tools via JSON =====
system_prompt = """
You are CopilotAI, a smart business assistant.

You have these tools available:
1. get_current_date — gets todays date. No arguments needed.
2. calculate — does math. Needs: expression (string like "150*45")
3. generate_email — writes an email. Needs: client_name, purpose, tone

HOW TO USE A TOOL:
If you need a tool, respond ONLY with this exact JSON format and nothing else:
{"tool": "tool_name_here", "args": {"arg1": "value1"}}

Examples:
{"tool": "get_current_date", "args": {}}
{"tool": "calculate", "args": {"expression": "150*45"}}
{"tool": "generate_email", "args": {"client_name": "John", "purpose": "pending invoice", "tone": "formal"}}

After you get the tool result, give the user a clean helpful final answer in plain text.
If you do NOT need a tool, just reply normally in plain text.
"""

# ===== AGENT LOOP =====
def chat_with_agent(user_message):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message}
    ]

    while True:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages
        )

        reply = response.choices[0].message.content.strip()

        # Check if agent wants to use a tool
        try:
            # Try to parse reply as JSON
            parsed = json.loads(reply)

            if "tool" in parsed:
                tool_name = parsed["tool"]
                tool_args = parsed.get("args", {})

                print(f"🔧 Using tool: {tool_name} with {tool_args}")

                # Run the tool
                tool_result = execute_tool(tool_name, tool_args)
                print(f"✅ Tool result: {tool_result}")

                # Give result back to agent
                messages.append({"role": "assistant", "content": reply})
                messages.append({"role": "user",      "content": f"Tool result: {tool_result}. Now give the user a clean final answer."})

        # Not JSON — it's a normal final answer
        except json.JSONDecodeError:
            return reply

# ===== TEST =====
print(chat_with_agent("What is today's date?"))
print("---")
print(chat_with_agent("If I sell 150 products at $45 each what is my total revenue?"))
print("---")
print(chat_with_agent("Write a follow up email to John about his pending invoice, keep it formal"))

🔧 Using tool: get_current_date with {}
✅ Tool result: Sunday, May 10, 2026
Today's date is Sunday, May 10, 2026.
---
🔧 Using tool: calculate with {'expression': '150*45'}
✅ Tool result: 6750
Your total revenue from selling 150 products at $45 each is $6750.
---
🔧 Using tool: generate_email with {'client_name': 'John', 'purpose': 'pending invoice', 'tone': 'formal'}
✅ Tool result: Subject: Follow Up — pending invoice

Dear John,

I hope this message finds you well.
I wanted to follow up regarding pending invoice.
Please let me know if you need anything from our side.

Best regards,
[Your Name]
Here is a follow-up email to John about his pending invoice:

Subject: Follow Up — pending invoice

Dear John,

I hope this message finds you well.
I wanted to follow up regarding pending invoice.
Please let me know if you need anything from our side.

Best regards,
[Your Name]


In [ ]:
from google.colab import userdata
from groq import Groq
import json
import datetime

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

# ===== MEMORY FUNCTIONS =====

MEMORY_FILE = "memory.json"

def load_memory():
    """Load memory from file"""
    try:
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    except:
        # No memory file yet — start fresh
        return {"user_name": "", "business_type": "", "preferences": [], "past_topics": []}

def save_memory(memory):
    """Save memory to file"""
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f, indent=2)
    print(f"💾 Memory saved!")

def update_memory(memory, key, value):
    """Update a specific memory field"""
    if key in ["preferences", "past_topics"]:
        if value not in memory[key]:
            memory[key].append(value)
    else:
        memory[key] = value
    save_memory(memory)
    return memory

# ===== TOOLS =====
def get_current_date():
    return datetime.datetime.now().strftime("%A, %B %d, %Y")

def calculate(expression):
    try:
        result = eval(expression)
        return f"{result}"
    except:
        return "Invalid calculation"

def generate_email(client_name, purpose, tone):
    return f"""Subject: Follow Up — {purpose}

Dear {client_name},

I hope this message finds you well.
I wanted to follow up regarding {purpose}.
Please let me know if you need anything from our side.

Best regards,
[Your Name]"""

def execute_tool(tool_name, tool_args, memory):
    if tool_name == "get_current_date":
        return get_current_date()
    elif tool_name == "calculate":
        return calculate(tool_args["expression"])
    elif tool_name == "generate_email":
        return generate_email(
            tool_args["client_name"],
            tool_args["purpose"],
            tool_args["tone"]
        )
    elif tool_name == "remember":
        # Save something to long term memory
        memory = update_memory(memory, tool_args["key"], tool_args["value"])
        return f"Remembered: {tool_args['key']} = {tool_args['value']}"
    return "Tool not found"

# ===== SYSTEM PROMPT WITH MEMORY =====
def build_system_prompt(memory):
    memory_context = f"""
WHAT YOU KNOW ABOUT THE USER:
- Name: {memory['user_name'] or 'Unknown'}
- Business type: {memory['business_type'] or 'Unknown'}
- Preferences: {', '.join(memory['preferences']) or 'None yet'}
- Past topics discussed: {', '.join(memory['past_topics']) or 'None yet'}
"""

    return f"""
You are CopilotAI, a smart business assistant with memory.

{memory_context}

Use this information to personalize your responses.
If user tells you their name or business type, save it using the remember tool.

You have these tools available:
1. get_current_date — gets todays date. No arguments needed.
2. calculate — does math. Needs: expression (string like "150*45")
3. generate_email — writes an email. Needs: client_name, purpose, tone
4. remember — saves important info. Needs: key (user_name/business_type/preferences/past_topics), value

HOW TO USE A TOOL:
Respond ONLY with this exact JSON format and nothing else:
{{"tool": "tool_name_here", "args": {{"arg1": "value1"}}}}

Examples:
{{"tool": "get_current_date", "args": {{}}}}
{{"tool": "remember", "args": {{"key": "user_name", "value": "John"}}}}
{{"tool": "remember", "args": {{"key": "past_topics", "value": "invoice follow up"}}}}

After tool result, give the user a clean helpful final answer in plain text.
If no tool needed, reply normally in plain text.
"""

# ===== AGENT LOOP WITH MEMORY =====
def chat_with_agent(user_message, conversation_history, memory):
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    messages = [
        {"role": "system", "content": build_system_prompt(memory)}
    ] + conversation_history

    while True:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages
        )

        reply = response.choices[0].message.content.strip()

        try:
            parsed = json.loads(reply)

            if "tool" in parsed:
                tool_name = parsed["tool"]
                tool_args = parsed.get("args", {})

                print(f"🔧 Using tool: {tool_name} with {tool_args}")

                tool_result = execute_tool(tool_name, tool_args, memory)
                print(f"✅ Tool result: {tool_result}")

                messages.append({"role": "assistant", "content": reply})
                messages.append({"role": "user", "content": f"Tool result: {tool_result}. Now give the user a clean final answer."})

        except json.JSONDecodeError:
            conversation_history.append({
                "role": "assistant",
                "content": reply
            })
            return reply, conversation_history, memory

# ===== START A CONVERSATION =====

# Load memory at start
memory = load_memory()
conversation_history = []

print("=" * 50)
print("🤖 CopilotAI — Business Assistant with Memory")
print("=" * 50)

# Test 1
print("You: Hi! My name is Ahmed and I run a clothing business")
response, conversation_history, memory = chat_with_agent(
    "Hi! My name is Ahmed and I run a clothing business",
    conversation_history, memory
)
print(f"🤖 Agent: {response}")
print()

# Test 2
print("You: What is my total revenue if I sell 200 jackets at $80 each?")
response, conversation_history, memory = chat_with_agent(
    "What is my total revenue if I sell 200 jackets at $80 each?",
    conversation_history, memory
)
print(f"🤖 Agent: {response}")
print()

# Simulate new conversation
print("=" * 50)
print("🔄 New conversation starting — reloading memory...")
print("=" * 50)

memory = load_memory()
conversation_history = []

print("You: Hey do you remember me?")
response, conversation_history, memory = chat_with_agent(
    "Hey do you remember me?",
    conversation_history, memory
)
print(f"🤖 Agent: {response}")

# Show what's saved in memory
print()
print("=" * 50)
print("💾 What agent remembers:")
print(json.dumps(memory, indent=2))

🤖 CopilotAI — Business Assistant with Memory
You: Hi! My name is Ahmed and I run a clothing business
🤖 Agent: Hello Ahmed, it's nice to meet you. I've taken note of your name and business type.

{"tool": "remember", "args": {"key": "user_name", "value": "Ahmed"}}
{"tool": "remember", "args": {"key": "business_type", "value": "clothing business"}}

I'd be happy to help you with any questions or concerns you may have about your clothing business. What can I assist you with today?

You: What is my total revenue if I sell 200 jackets at $80 each?
🤖 Agent: To calculate your total revenue, I'll multiply the number of jackets sold by the price per jacket.

{"tool": "calculate", "args": {"expression": "200 * 80"}}

Your total revenue is $16000. If you'd like to discuss how to increase sales or optimize pricing, I'm here to help.

🔄 New conversation starting — reloading memory...
You: Hey do you remember me?
🤖 Agent: No, I don't remember you. This is our first interaction, and I don't have any 

In [ ]:
from google.colab import userdata
from groq import Groq
import json
import datetime
import re

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

# ===== MEMORY FUNCTIONS =====
MEMORY_FILE = "memory.json"

def load_memory():
    try:
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    except:
        return {"user_name": "", "business_type": "", "preferences": [], "past_topics": []}

def save_memory(memory):
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f, indent=2)

def update_memory(memory, key, value):
    if key in ["preferences", "past_topics"]:
        if value not in memory[key]:
            memory[key].append(value)
    else:
        memory[key] = value
    save_memory(memory)
    return memory

# ===== TOOLS =====
def get_current_date():
    return datetime.datetime.now().strftime("%A, %B %d, %Y")

def calculate(expression):
    try:
        result = eval(expression)
        return f"{result}"
    except:
        return "Invalid calculation"

def generate_email(client_name, purpose, tone):
    return f"""Subject: Follow Up — {purpose}

Dear {client_name},

I hope this message finds you well.
I wanted to follow up regarding {purpose}.
Please let me know if you need anything from our side.

Best regards,
[Your Name]"""

def execute_tool(tool_name, tool_args, memory):
    if tool_name == "get_current_date":
        return get_current_date()
    elif tool_name == "calculate":
        return calculate(tool_args["expression"])
    elif tool_name == "generate_email":
        return generate_email(
            tool_args["client_name"],
            tool_args["purpose"],
            tool_args["tone"]
        )
    elif tool_name == "remember":
        memory = update_memory(memory, tool_args["key"], tool_args["value"])
        return f"Remembered: {tool_args['key']} = {tool_args['value']}"
    return "Tool not found"

# ===== EXTRACT AND RUN ALL TOOL CALLS IN A REPLY =====
def extract_and_run_tools(reply, memory):
    """Find all JSON tool calls inside a reply and execute them"""
    # Find all JSON blocks in the text
    json_pattern = r'\{[^{}]*"tool"[^{}]*\}'
    matches = re.findall(json_pattern, reply)

    for match in matches:
        try:
            parsed = json.loads(match)
            if "tool" in parsed:
                tool_name = parsed["tool"]
                tool_args = parsed.get("args", {})
                print(f"💾 Saving to memory: {tool_name} → {tool_args}")
                execute_tool(tool_name, tool_args, memory)
        except:
            pass

    # Clean the reply — remove all JSON tool calls from text
    clean_reply = re.sub(json_pattern, "", reply).strip()
    # Clean up extra newlines
    clean_reply = re.sub(r'\n{3,}', '\n\n', clean_reply).strip()
    return clean_reply, memory

# ===== SYSTEM PROMPT =====
def build_system_prompt(memory):
    memory_context = f"""
WHAT YOU KNOW ABOUT THE USER:
- Name: {memory['user_name'] or 'Unknown'}
- Business type: {memory['business_type'] or 'Unknown'}
- Preferences: {', '.join(memory['preferences']) or 'None yet'}
- Past topics discussed: {', '.join(memory['past_topics']) or 'None yet'}
"""
    return f"""
You are CopilotAI, a smart business assistant with memory.

{memory_context}

Use this information to personalize your responses.
Greet the user by name if you know it.

You have these tools:
1. get_current_date — gets todays date
2. calculate — does math. Needs: expression
3. generate_email — writes email. Needs: client_name, purpose, tone
4. remember — saves info. Needs: key (user_name/business_type/preferences/past_topics), value

IMPORTANT RULES:
- If user shares their name → immediately use remember tool to save user_name
- If user shares business type → immediately use remember tool to save business_type
- To use a tool respond with ONLY this JSON on its own line:
{{"tool": "tool_name", "args": {{"key": "value"}}}}
- After saving to memory, confirm to the user in plain text
- For calculate or generate_email use the tool then give final answer
"""

# ===== AGENT LOOP =====
def chat_with_agent(user_message, conversation_history, memory):
    conversation_history.append({"role": "user", "content": user_message})

    messages = [
        {"role": "system", "content": build_system_prompt(memory)}
    ] + conversation_history

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages
    )

    reply = response.choices[0].message.content.strip()

    # Extract and run any tool calls found in the reply
    clean_reply, memory = extract_and_run_tools(reply, memory)

    conversation_history.append({"role": "assistant", "content": clean_reply})
    return clean_reply, conversation_history, memory

# ===== RUN =====
memory = load_memory()
conversation_history = []

print("=" * 50)
print("🤖 CopilotAI — Business Assistant with Memory")
print("=" * 50)

print("You: Hi! My name is Ahmed and I run a clothing business")
response, conversation_history, memory = chat_with_agent(
    "Hi! My name is Ahmed and I run a clothing business",
    conversation_history, memory
)
print(f"🤖 Agent: {response}\n")

print("You: What is my revenue if I sell 200 jackets at $80 each?")
response, conversation_history, memory = chat_with_agent(
    "What is my revenue if I sell 200 jackets at $80 each?",
    conversation_history, memory
)
print(f"🤖 Agent: {response}\n")

# Simulate new conversation
print("=" * 50)
print("🔄 New conversation — reloading memory...")
print("=" * 50)

memory = load_memory()
conversation_history = []

print("You: Hey do you remember me?")
response, conversation_history, memory = chat_with_agent(
    "Hey do you remember me?",
    conversation_history, memory
)
print(f"🤖 Agent: {response}\n")

print("💾 Saved memory:")
print(json.dumps(memory, indent=2))

🤖 CopilotAI — Business Assistant with Memory
You: Hi! My name is Ahmed and I run a clothing business
🤖 Agent: {"tool": "remember", "args": {"key": "user_name", "value": "Ahmed"}}
{"tool": "remember", "args": {"key": "business_type", "value": "clothing business"}}
Hello Ahmed, I've taken note of your name and business type.

You: What is my revenue if I sell 200 jackets at $80 each?
🤖 Agent: {"tool": "calculate", "args": {"expression": "200 * 80"}}
Your revenue is $16000.

🔄 New conversation — reloading memory...
You: Hey do you remember me?
🤖 Agent: No, I don't remember you. This is our first conversation, and I don't have any information about you yet. What's your name?

💾 Saved memory:
{
  "user_name": "",
  "business_type": "",
  "preferences": [],
  "past_topics": []
}


In [ ]:
from google.colab import userdata
from groq import Groq
import json
import datetime
import gradio as gr
import re

api_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=api_key)

# ===== TOOLS =====
def get_current_date():
    return datetime.datetime.now().strftime("%A, %B %d, %Y")

def calculate(expression):
    try:
        result = eval(expression)
        return f"{result}"
    except:
        return "Invalid calculation"

def generate_email(client_name, purpose, tone):
    return f"""Subject: Follow Up — {purpose}

Dear {client_name},

I hope this message finds you well.
I wanted to follow up regarding {purpose}.
Please let me know if you need anything from our side.

Best regards,
[Your Name]"""

def execute_tool(tool_name, tool_args):
    if tool_name == "get_current_date":
        return get_current_date()
    elif tool_name == "calculate":
        return calculate(tool_args.get("expression", ""))
    elif tool_name == "generate_email":
        return generate_email(
            tool_args.get("client_name", "Client"),
            tool_args.get("purpose", ""),
            tool_args.get("tone", "formal")
        )
    return "Tool not found"

# ===== SYSTEM PROMPT =====
system_prompt = """
You are CopilotAI, a smart business assistant.

You have these tools:
1. get_current_date — gets todays date. No args needed.
2. calculate — does math. Needs: expression (like "150*45")
3. generate_email — writes email. Needs: client_name, purpose, tone

To use a tool respond ONLY with this exact JSON and nothing else:
{"tool": "tool_name", "args": {"key": "value"}}

After getting tool result give the user a clean helpful answer.
If no tool needed just reply normally.
Always be professional and concise.
"""

# ===== AGENT =====
def agent_reply(user_message, history):
    # Build messages from history
    messages = [{"role": "system", "content": system_prompt}]

    for human, assistant in history:
        messages.append({"role": "user",      "content": human})
        messages.append({"role": "assistant", "content": assistant})

    messages.append({"role": "user", "content": user_message})

    # Agent loop
    while True:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages
        )

        reply = response.choices[0].message.content.strip()

        try:
            parsed = json.loads(reply)
            if "tool" in parsed:
                tool_name = parsed["tool"]
                tool_args = parsed.get("args", {})
                tool_result = execute_tool(tool_name, tool_args)

                messages.append({"role": "assistant", "content": reply})
                messages.append({"role": "user",      "content": f"Tool result: {tool_result}. Now give the user a clean final answer."})
            else:
                return reply

        except json.JSONDecodeError:
            return reply

# ===== GRADIO UI =====
with gr.Blocks(theme=gr.themes.Soft(), title="CopilotAI") as demo:

    gr.Markdown("""
    # 🤖 CopilotAI — Your Business Assistant
    ### Ask me anything about your business!
    """)

    chatbot = gr.Chatbot(
        label="CopilotAI",
        height=450,
        show_label=False
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask me anything... e.g. Calculate my profit, Write an email, What's today's date",
            label="Your message",
            scale=4
        )
        send_btn = gr.Button("Send 🚀", scale=1, variant="primary")

    with gr.Row():
        gr.Examples(
            examples=[
                "What is today's date?",
                "Calculate my revenue: 500 units at $30 each",
                "Write a follow up email to Sarah about her pending payment, keep it formal",
                "What can you help me with?"
            ],
            inputs=msg,
            label="💡 Try these examples"
        )

    clear_btn = gr.Button("🗑️ Clear Chat", variant="secondary")

    # ===== ACTIONS =====
    def respond(user_message, chat_history):
        if not user_message.strip():
            return "", chat_history
        reply = agent_reply(user_message, chat_history)
        chat_history.append((user_message, reply))
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    send_btn.click(respond, [msg, chatbot], [msg, chatbot])
    clear_btn.click(lambda: [], None, chatbot)

demo.launch(share=True)

/tmp/ipykernel_3264/3373737492.py:100: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="CopilotAI") as demo:
/tmp/ipykernel_3264/3373737492.py:107: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_3264/3373737492.py:107: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cfaac0ba50528ac4ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
